# Veri Keşfi, Özellik Mühendisliği ve Model Eğitimi

Bu notebook'un amacı, topladığımız ham verileri işlemek, yeni özellikler türetmek ve bu verilerle bir makine öğrenmesi modeli eğitmektir.

## 1. Kütüphanelerin Yüklenmesi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import plotly.express as px # İnteraktif grafikler için Plotly kütüphanesini ekliyoruz
import pandas_ta as ta # Teknik analiz göstergeleri için pandas-ta kütüphanesini ekliyoruz
import optuna # Hiperparametre optimizasyonu için Optuna kütüphanesini ekliyoruz

# Grafiklerin daha güzel görünmesi için ayar
sns.set_style('whitegrid')

## 2. Veri Setlerinin Yüklenmesi ve Birleştirilmesi

In [ ]:
# Piyasa verisini yükle
btc_market_path = os.path.join('..', 'data', 'raw', 'BTC_USDT_1d_market_data.csv')
df_btc = pd.read_csv(btc_market_path, index_col=0, parse_dates=True)

# Korku & Açgözlülük Endeksi verisini yükle
fear_greed_path = os.path.join('..', 'data', 'raw', 'fear_greed_index.csv')
df_fg = pd.read_csv(fear_greed_path, index_col='date', parse_dates=True)

# İki DataFrame'i tarihe göre birleştir
df_merged = df_btc.join(df_fg, how='inner')

print("Birleştirilmiş Veri Seti:")
df_merged.head()

## 3. Özellik Mühendisliği (Feature Engineering)

In [ ]:
# pandas-ta kütüphanesini kullanarak tüm teknik göstergeleri tek seferde hesapla
df_merged.ta.rsi(length=14, append=True)
df_merged.ta.macd(fast=12, slow=26, signal=9, append=True)
df_merged.ta.bbands(length=20, std=2, append=True)

# Hareketli ortalamalar ve getiri
df_merged['daily_return'] = df_merged['close'].pct_change()
df_merged['MA7'] = df_merged['close'].rolling(window=7).mean()
df_merged['MA30'] = df_merged['close'].rolling(window=30).mean()

print("Teknik Göstergeler Eklenmiş Veri Seti:")
df_merged.tail()

## 4. Hedef Değişkeni (Target Label) Oluşturma

In [ ]:
threshold = 0.005 # %0.5'lik eşik değer
future_close = df_merged['close'].shift(-1)
df_merged['target'] = (future_close > df_merged['close'] * (1 + threshold)).astype(int)

# Oluşan boş (NaN) satırları temizle
df_merged.dropna(inplace=True)

print("Hedef Değişkeni Eklenmiş Son Veri Seti:")
df_merged[['close', 'target']].tail()

## 5. Baseline Model: Sadece Piyasa Verileri

İlk olarak, sadece piyasa ve korku endeksi verilerini kullanarak bir temel model eğitelim ve performansını ölçelim.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Özellikler (X) ve hedef (y) değişkenlerini ayıralım
X = df_merged.drop(columns=['target', 'fear_greed_classification'])
y = df_merged['target']

# Veriyi zamana göre ayırma (%80 eğitim, %20 test)
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Baseline Modeli Eğitme
print("Baseline Model eğitiliyor...")
baseline_model = XGBClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
print("Eğitim tamamlandı.")

# Performansı Değerlendirme
y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
print(f"\nBaseline Modelin Doğruluk Oranı (Accuracy): {accuracy_baseline:.4f}")
print("\nBaseline Model Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred_baseline))

## 6. Reddit Verileri ile Zenginleştirme: Duygu Analizi

Şimdi, topladığımız Reddit gönderilerinin başlıklarına duygu analizi uygulayarak yeni bir özellik oluşturacağız. Bu, modelimize topluluk hissiyatı hakkında bilgi verecektir.

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Reddit verisini yükle
reddit_path = os.path.join('..', 'data', 'raw', 'BTC_reddit_data.csv')
df_reddit = pd.read_csv(reddit_path)

# Tarih sütununu doğru formata çevir ve saat bilgisini at
df_reddit['created_utc'] = pd.to_datetime(df_reddit['created_utc']).dt.date
df_reddit.rename(columns={'created_utc': 'date'}, inplace=True)
df_reddit['date'] = pd.to_datetime(df_reddit['date'])

# Duygu analizi modelini yükle (ilk çalıştırmada internetten indirilir)
print("Duygu analizi modeli yükleniyor... (Bu işlem birkaç dakika sürebilir)")
sentiment_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model yüklendi.")

# Gönderi başlıklarını (title) listeye çevir
titles = df_reddit['title'].tolist()

# Her başlık için duygu skorunu hesapla (bu işlem de zaman alabilir)
print("Başlıklar için duygu skorları hesaplanıyor...")
embeddings = sentiment_model.encode(titles, convert_to_tensor=True)

# Basit bir yaklaşım: Pozitif ve negatif terimlerle karşılaştırma
pos_embedding = sentiment_model.encode(['positive hopeful optimistic good great buy rocket moon'], convert_to_tensor=True)
neg_embedding = sentiment_model.encode(['negative pessimistic bearish bad terrible sell dump crash'], convert_to_tensor=True)

pos_scores = util.cos_sim(embeddings, pos_embedding).numpy().flatten()
neg_scores = util.cos_sim(embeddings, neg_embedding).numpy().flatten()

# Net duygu skorunu hesapla (pozitif - negatif)
df_reddit['sentiment_score'] = pos_scores - neg_scores

print("Duygu skorları hesaplandı.")

### Günlük Ortalama Duygu Skorunu Hesaplama

In [ ]:
# Her gün için ortalama duygu skorunu hesapla
daily_sentiment = df_reddit.groupby('date')['sentiment_score'].mean().to_frame(name='reddit_sentiment')

daily_sentiment.head()

## 7. Gelişmiş Model: Reddit Duygu Verisi ile Yeniden Eğitim

Şimdi, ana veri setimize günlük Reddit duygu skorlarını ekleyerek modelimizi yeniden eğiteceğiz ve performans artışını gözlemleyeceğiz.

In [ ]:
# Ana veri setine duygu skorlarını ekle
df_final = df_merged.join(daily_sentiment, how='left')

# ChainedAssignmentError uyarısını önlemek için daha güvenli .loc kullanımı
# Duygu skorları her gün olmayabilir, bu yüzden boş kalan günleri doldur.
df_final.loc[:, 'reddit_sentiment'] = df_final['reddit_sentiment'].ffill()
df_final.loc[:, 'reddit_sentiment'] = df_final['reddit_sentiment'].fillna(0)

df_final.dropna(inplace=True)

# Yeni özellik seti (X) ve hedef (y) oluştur
X_final = df_final.drop(columns=['target', 'fear_greed_classification'])
y_final = df_final['target']

# Veriyi son haline göre yeniden ayır
split_index_final = int(len(X_final) * 0.8)
X_train_final, X_test_final = X_final[:split_index_final], X_final[split_index_final:]
y_train_final, y_test_final = y_final[:split_index_final], y_final[split_index_final:]

# Gelişmiş Modeli Eğit
print("Gelişmiş Model (Reddit verisiyle) eğitiliyor...")
advanced_model = XGBClassifier(random_state=42, base_score=0.5)
advanced_model.fit(X_train_final, y_train_final)
print("Eğitim tamamlandı.")

# Performansı Değerlendir
y_pred_advanced = advanced_model.predict(X_test_final)
accuracy_advanced = accuracy_score(y_test_final, y_pred_advanced)
print(f"\nGelişmiş Modelin Doğruluk Oranı (Accuracy): {accuracy_advanced:.4f}")

if len(y_test_final.unique()) < 2:
    print("\nUyarı: Test setinde sadece tek bir sınıf bulundu, bu nedenle sınıflandırma raporu oluşturulamıyor.")
else:
    print("\nGelişmiş Model Sınıflandırma Raporu:")
    print(classification_report(y_test_final, y_pred_advanced))

## 8. Model İyileştirme (Hiperparametre Optimizasyonu)

Optuna kütüphanesini kullanarak XGBoost modelimiz için en iyi hiperparametreleri (ayarları) bulacağız. Bu işlem, modelin doğruluk oranını artırabilir.

In [ ]:
def objective(trial):
    """Optuna için amaç fonksiyonu. Her denemede farklı parametrelerle model eğitir ve doğruluk oranını döndürür."""
    
    # Test edilecek hiperparametre aralıklarını tanımla
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5)
    }
    
    model = XGBClassifier(**param, random_state=42)
    model.fit(X_train_final, y_train_final)
    preds = model.predict(X_test_final)
    accuracy = accuracy_score(y_test_final, preds)
    return accuracy

# Optuna çalışmasını (study) oluştur ve en iyi doğruluğu bulmaya çalış
study = optuna.create_study(direction='maximize')
# n_trials: Kaç farklı parametre kombinasyonunun deneneceği. Daha yüksek daha iyi sonuç verebilir ama daha uzun sürer.
print("Hiperparametre optimizasyonu başlıyor... (Bu işlem birkaç dakika sürebilir)")
study.optimize(objective, n_trials=50)
print("Optimizasyon tamamlandı.")

print("En iyi deneme:")
trial = study.best_trial
print(f"  Değer (Accuracy): {trial.value}")
print("  En İyi Parametreler: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

## 9. Optimize Edilmiş Final Model

Optuna'nın bulduğu en iyi parametrelerle son modelimizi eğitelim ve performansını değerlendirelim.

In [ ]:
best_params = study.best_params
optimized_model = XGBClassifier(**best_params, random_state=42)

print("Optimize edilmiş model eğitiliyor...")
optimized_model.fit(X_train_final, y_train_final)
print("Eğitim tamamlandı.")

y_pred_optimized = optimized_model.predict(X_test_final)
accuracy_optimized = accuracy_score(y_test_final, y_pred_optimized)

print(f"\nOptimize Edilmiş Modelin Doğruluk Oranı (Accuracy): {accuracy_optimized:.4f}")
print("\nOptimize Edilmiş Model Sınıflandırma Raporu:")
print(classification_report(y_test_final, y_pred_optimized))

## 10. Final Modelin Açıklanabilirliği (SHAP)

Son olarak, en iyi ayarları bulduğumuz optimize edilmiş modelimizin kararlarını verirken hangi özelliklere daha fazla önem verdiğini SHAP ile analiz edelim.

In [ ]:
import shap

# Optimize edilmiş modelimiz için bir SHAP açıklayıcı oluşturalım
explainer = shap.TreeExplainer(optimized_model)

# Test verilerimiz için SHAP değerlerini hesaplayalım
shap_values = explainer.shap_values(X_test_final)

print("SHAP değerleri hesaplandı. Özet grafiği oluşturuluyor...")

# Özelliklerin genel önemini gösteren özet grafiği
shap.summary_plot(shap_values, X_test_final, plot_type="bar")

## 11. Final Modelini Kaydetme

Eğittiğimiz ve optimize ettiğimiz en iyi modeli, API'mizde kullanmak üzere bir dosyaya kaydedelim. Bu işlem için daha önce oluşturduğumuz `model_utils.py` içindeki `save_model` fonksiyonunu kullanacağız.

In [ ]:
# model_utils script'ini import edebilmek için sistem yoluna projenin ana dizinini ekliyoruz
import sys
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.model_utils import save_model

# Optimize edilmiş en iyi modelimizi kaydedelim
save_model(optimized_model, model_name="btc_xgboost_v1")

## 12. Kayıt İşlemini Doğrulama

Modelin doğru yere kaydedildiğinden emin olmak için, projenin ana dizinindeki 'models' klasörünün içeriğini listeleyelim.

In [ ]:
project_root = os.path.abspath(os.path.join('..'))
models_dir = os.path.join(project_root, 'models')

print(f"'models' klasörünün tam yolu: {models_dir}")

if os.path.exists(models_dir):
    print("'models' klasörünün içeriği:")
    print(os.listdir(models_dir))
else:
    print("HATA: 'models' klasörü bulunamadı!")